# Video loop

Testing out how to use a modified Qwen2.5-VL/Qwen3-VL model to stream inference.


## Latest Qwen3 models

- [Qwen3-VL 8B Instruct](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct)
- [Qwen3-VL 4B Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct)

In [ ]:
import time
from typing import Any

import cv2
import torch
from PIL import Image
from transformers import (
    AutoModel,
    AutoModelForVision2Seq,
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
)

MODEL_CONFIGS = {
    "qwen3b": {
        "id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "loader": Qwen2_5_VLForConditionalGeneration,
    },
    "smolvlm": {
        "id": "HuggingFaceTB/SmolVLM-500M-Instruct",
        "loader": AutoModelForVision2Seq,
    },
}

current_model_key = "smolvlm"


def load_model_and_processor(key: str) -> tuple[AutoModel, AutoProcessor]:
    model_cfg = MODEL_CONFIGS[key]
    print(f"Loading model: {model_cfg['id']}...")
    model: AutoModel = model_cfg["loader"].from_pretrained(
        model_cfg["id"],
        device_map="auto",
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    processor: AutoProcessor = AutoProcessor.from_pretrained(model_cfg["id"])
    print("Model loaded!")
    return model, processor


def load_camera_source() -> cv2.VideoCapture:
    # Start capturing video
    cap = cv2.VideoCapture(0)  # 0 is the default webcam
    if not cap.isOpened():
        raise OSError("Cannot open webcam")
    return cap


def analyze_frame(
    model: Any, processor: AutoProcessor, image: Image.Image, prompt: str
) -> str:
    # Prepare the input for the model
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(
        model.device
    )
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_text: str = processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0]
    return generated_text


model, processor = load_model_and_processor(current_model_key)
cap = load_camera_source()

frame_count = 0
inference_interval = 120
prompt = "Describe what you see"
frame_buffer = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Display live video feed
    cv2.imshow("IRIS Live Stream - Press Q to quit", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break

    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frame_buffer.append(pil_image)

    # Run inference periodically
    if frame_count % inference_interval == 0:
        recent_frames = frame_buffer[-8:]

        print(f"Analyzing frame {frame_count} ({current_model_key})")

        output_text = analyze_frame(model, processor, pil_image, prompt)

        print(f"{current_model_key} output: {output_text}")

        frame_buffer.clear()

cap.release()
cv2.destroyAllWindows()
print("\nStream ended")

Loading model: HuggingFaceTB/SmolVLM-500M-Instruct...


KeyboardInterrupt: 